In [32]:
import pandas as pd

In [33]:
!pip install transformers
!pip install numpy
!pip install torch
!pip install scikit-learn
!pip install datasets


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch

In [35]:
# -----------------------
# 1) Paths (change these)
# -----------------------
TRAIN_PATH = "testing/train.csv"
VAL_PATH   = "testing/validation.csv"
TEST_PATH  = "testing/test.csv"

In [36]:
# -----------------------
# 2) Load CSVs (NO header)
#   Col0 = headline text
#   Col1 = label in {-1, 0, 1}
# -----------------------
def load_noheader_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, header=None)
    df.columns = ["text", "label"]
    df = df.dropna(subset=["text", "label"]).copy()
    df["text"] = df["text"].astype(str).str.strip()
    # labels sometimes load as float; force int
    df["label"] = df["label"].astype(int)
    return df

df_train = load_noheader_csv(TRAIN_PATH)
df_val   = load_noheader_csv(VAL_PATH)
df_test  = load_noheader_csv(TEST_PATH)

print("Train/Val/Test sizes:", len(df_train), len(df_val), len(df_test))
print("Train label counts:\n", df_train["label"].value_counts().sort_index())


Train/Val/Test sizes: 61692 1000 1000
Train label counts:
 -1    13037
 0    24033
 1    24622
Name: label, dtype: int64


In [37]:
# -----------------------
# 3) Map labels {-1,0,1} -> {0,1,2}
#   0=negative,1=neutral,2=positive (stable mapping)
# -----------------------
label_map = {-1: 0, 0: 1, 1: 2}
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}
label_to_id = {v: k for k, v in id_to_label.items()}

def apply_label_map(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["label_id"] = df["label"].map(label_map)
    if df["label_id"].isna().any():
        bad = df[df["label_id"].isna()]["label"].unique()
        raise ValueError(f"Found labels not in {-1,0,1}: {bad}")
    return df

df_train = apply_label_map(df_train)
df_val   = apply_label_map(df_val)
df_test  = apply_label_map(df_test)

In [38]:
# -----------------------
# 4) Convert to HF Datasets
#   Trainer expects column name "labels"
# -----------------------
train_ds = Dataset.from_pandas(df_train[["text", "label_id"]].rename(columns={"label_id": "labels"}))
val_ds   = Dataset.from_pandas(df_val[["text", "label_id"]].rename(columns={"label_id": "labels"}))
test_ds  = Dataset.from_pandas(df_test[["text", "label_id"]].rename(columns={"label_id": "labels"}))

# -----------------------

In [39]:
# -----------------------
# 5) Tokenizer + tokenize function
# -----------------------
model_name = "yiyanghkust/finbert-pretrain"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

Map:   0%|          | 0/61692 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [40]:
# -----------------------
# 6) Model
# -----------------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id
)

# -----------------------
# 7) Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-pretrain and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
!pip install -U "transformers[torch]"
!pip install -U "accelerate>=0.26.0"



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import sys
import accelerate
import transformers
from transformers.utils import is_accelerate_available, is_torch_available

print("Python:", sys.executable)
print("accelerate:", accelerate.__version__, "file:", accelerate.__file__)
print("transformers:", transformers.__version__)
print("is_accelerate_available():", is_accelerate_available())
print("is_torch_available():", is_torch_available())

try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
except Exception as e:
    print("torch import failed:", repr(e))


Python: c:\Users\anasa\AppData\Local\Programs\Python\Python311\python.exe
accelerate: 1.12.0 file: c:\Users\anasa\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\__init__.py
transformers: 4.57.6
is_accelerate_available(): False
is_torch_available(): True
torch: 2.6.0+cpu cuda: False


In [44]:
import importlib
import transformers.utils.import_utils as iu

importlib.reload(iu)

from transformers.utils import is_accelerate_available
print("After reload, is_accelerate_available():", is_accelerate_available())


After reload, is_accelerate_available(): True


In [ ]:
import accelerate, transformers
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)



# -----------------------
# 8) TrainingArguments + Trainer
# -----------------------
import importlib
import transformers.utils.import_utils as iu
importlib.reload(iu)

from transformers import TrainingArguments, Trainer
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

args = TrainingArguments(
    output_dir="models/finbert_my_malaysia",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=False,  # CPU training → keep False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

pred = trainer.predict(test_ds)
test_preds = np.argmax(pred.predictions, axis=1)

label_order = ["negative", "neutral", "positive"]
print(classification_report(pred.label_ids, test_preds, target_names=label_order))
print(confusion_matrix(pred.label_ids, test_preds))


# -----------------------
# 9) Evaluate on TEST
# -----------------------
pred = trainer.predict(test_ds)
test_logits = pred.predictions
test_labels = pred.label_ids
test_preds  = np.argmax(test_logits, axis=1)

label_order = ["negative", "neutral", "positive"]

print("\nClassification report (TEST):")
print(classification_report(test_labels, test_preds, target_names=label_order))

print("\nConfusion matrix (TEST):")
print(confusion_matrix(test_labels, test_preds))

# -----------------------
# 10) Save best model
# -----------------------
save_dir = "models/finbert_my_malaysia/best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("\nSaved to:", save_dir)

accelerate: 1.12.0
transformers: 4.57.6


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`